# Notebook 2 — Model Fitting & Statistical Tests

Fits the Pareto power law  Nₛ = C · A^α  to each fund via OLS in log-space.

**Methods implemented:**
- Fund-level OLS with HC1 heteroskedasticity-robust standard errors
- Non-parametric bootstrap CIs  (B = 5 000, with measurement-noise injection)
- AIC-based model selection: power law vs. linear vs. log-normal null
- Likelihood-ratio test (LRT) of AUM significance
- Leave-one-out cross-validation (LOO-CV)
- Pooled regression across all funds

**Pseudocode — OLS:**
```
x = log(AUM),   y = log(headcount)
α̂ = Σ(xᵢ-x̄)(yᵢ-ȳ) / Σ(xᵢ-x̄)²
Ĉ = exp(ȳ - α̂·x̄)
HC1 SE = sqrt( [Σ((xᵢ-x̄)·eᵢ)²] / [Σ(xᵢ-x̄)²]² · n/(n-2) )
```

**Pseudocode — bootstrap CI:**
```
for b in 1..B:
    idx   = resample with replacement
    y_b   = y[idx] + Normal(0, σ_meas)   # inject measurement noise
    α_b   = OLS(y_b, x[idx])
CI = percentile([α_b], [2.5, 97.5])
```


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

IS   = pd.read_csv('../data/insample_funds.csv')
OOS  = pd.read_csv('../data/oos_funds.csv')
META = pd.read_csv('../data/strategy_metadata.csv', index_col='strategy_code')
COLORS  = META['color'].to_dict()
MARKERS = META['marker'].to_dict()

np.random.seed(42)


## 2.1  Core estimators

In [ ]:
def ols_powerlaw(aum, hc):
    """
    OLS of log(hc) ~ log(aum) with HC1 robust SE.
    Returns dict: alpha, C, SE, R2, residuals, efficiency.
    """
    x = np.log(np.asarray(aum, float))
    y = np.log(np.asarray(hc,  float))
    n = len(x)
    xb, yb = x.mean(), y.mean()
    Sxx = ((x - xb)**2).sum()
    alpha = ((x - xb) * (y - yb)).sum() / Sxx
    lnC   = yb - alpha * xb
    C     = np.exp(lnC)
    e     = y - (lnC + alpha * x)
    # HC1 variance
    var_hc1 = ((x - xb)**2 * e**2).sum() / Sxx**2 * n / max(n-2, 1)
    r2 = 1 - e.var() / y.var() if y.var() > 1e-12 else np.nan
    eff = np.exp(xb - yb) * 1000   # AUM (bn) / headcount * 1000 = USD M
    return dict(alpha=alpha, C=C, lnC=lnC, SE=float(np.sqrt(max(var_hc1,0))),
                R2=r2, resid=e, x=x, y=y, n=n, efficiency=eff)


def bootstrap_ci(aum, hc, B=5000, sigma_meas=0.15, seed=42):
    """
    Percentile bootstrap CI for alpha with measurement noise injection.
    sigma_meas: 0.02 for audited Man Group, 0.15 otherwise.
    """
    rng = np.random.default_rng(seed)
    x = np.log(np.asarray(aum, float))
    y = np.log(np.asarray(hc,  float))
    n, alphas = len(x), []
    for _ in range(B):
        idx  = rng.integers(0, n, n)
        xi   = x[idx]
        yi   = y[idx] + rng.normal(0, sigma_meas, n)
        xb   = xi.mean()
        Sxx  = ((xi - xb)**2).sum()
        if Sxx < 1e-10: continue
        alphas.append(((xi - xb) * (yi - yi.mean())).sum() / Sxx)
    a = np.array(alphas)
    return (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))) if len(a) >= 100 else (np.nan, np.nan)


def loo_cv_mse(aum, hc):
    """Leave-one-out CV MSE in log-space.""    x = np.log(np.asarray(aum, float))
    y = np.log(np.asarray(hc,  float))
    n = len(x)
    if n < 3: return np.nan
    errs = []
    for i in range(n):
        xi, yi = np.delete(x,i), np.delete(y,i)
        xb = xi.mean(); Sxx = ((xi-xb)**2).sum()
        if Sxx < 1e-10: continue
        a  = ((xi-xb)*(yi-yi.mean())).sum()/Sxx
        lC = yi.mean() - a*xb
        errs.append((y[i] - (lC + a*x[i]))**2)
    return float(np.mean(errs))


def aic(x, y, model='powerlaw'):
    """AIC in log-space. model: 'powerlaw' (k=2) | 'lognormal' (k=1) | 'linear' (k=2)""    n = len(x)
    if model == 'powerlaw':
        xb=x.mean(); a=((x-xb)*(y-y.mean())).sum()/((x-xb)**2).sum()
        e = y - (y.mean()-a*xb + a*x); k=2
    elif model == 'lognormal':
        e = y - y.mean(); k=1
    else:  # linear on original scale — relative residuals
        A,N = np.exp(x), np.exp(y)
        b = ((A-A.mean())*(N-N.mean())).sum()/((A-A.mean())**2).sum()
        e = (N-(N.mean()-b*A.mean()+b*A))/N; k=2
    s2 = max((e**2).sum()/n, 1e-12)
    ll = -0.5*n*(np.log(2*np.pi*s2)+1)
    return 2*k - 2*ll


def lrt_p(x, y):
    """LRT: power law vs log-normal null (chi2_1).""    xb=x.mean(); Sxx=((x-xb)**2).sum()
    a=((x-xb)*(y-y.mean())).sum()/Sxx; lC=y.mean()-a*xb
    e_full = y-(lC+a*x); s2_full=max(e_full.var(), 1e-12)
    s2_null= max(y.var(), 1e-12); n=len(x)
    ll_full=-0.5*n*(np.log(2*np.pi*s2_full)+1)
    ll_null=-0.5*n*(np.log(2*np.pi*s2_null)+1)
    return float(stats.chi2.sf(max(2*(ll_full-ll_null),0), df=1))


## 2.2  Fit all in-sample funds

In [ ]:
# Man Group has audited data → lower noise injection in bootstrap
AUDITED_SIGMA = {'Man Group': 0.02}

fits = {}
for fund, g in IS.groupby('fund'):
    g = g.sort_values('year')
    aum, hc = g['aum_bn'].values, g['headcount'].values
    r = ols_powerlaw(aum, hc)
    sigma = AUDITED_SIGMA.get(fund, 0.15)
    lo, hi = bootstrap_ci(aum, hc, sigma_meas=sigma)
    r.update({
        'fund'    : fund,
        'strategy': g['strategy'].iloc[0],
        'aum'     : aum, 'hc': hc,
        'ci_lo'   : lo,  'ci_hi': hi,
        'ci_width': hi - lo,
        'weak_id' : (hi - lo) > 0.8,
        'aic_pl'  : aic(r['x'], r['y'], 'powerlaw'),
        'aic_ln'  : aic(r['x'], r['y'], 'lognormal'),
        'lrt_p'   : lrt_p(r['x'], r['y']),
        'loo_mse' : loo_cv_mse(aum, hc),
    })
    r['delta_aic'] = r['aic_pl'] - r['aic_ln']
    fits[fund] = r

# Pooled
all_aum = np.concatenate([r['aum'] for r in fits.values()])
all_hc  = np.concatenate([r['hc']  for r in fits.values()])
pool = ols_powerlaw(all_aum, all_hc)
print(f"Pooled:  α = {pool['alpha']:.3f},  C = {pool['C']:.0f},  R² = {pool['R2']:.3f},  n = {pool['n']}")


In [ ]:
# Pretty results table
rows = []
for f, r in sorted(fits.items(), key=lambda x: x[1]['alpha']):
    rows.append({'Fund':f, 'Str':r['strategy'], 'n':r['n'],
                 'alpha':round(r['alpha'],3), 'SE':round(r['SE'],3),
                 'CI_lo':round(r['ci_lo'],2), 'CI_hi':round(r['ci_hi'],2),
                 'C':int(r['C']), 'R2':round(r['R2'],3),
                 'LOO_MSE':round(r['loo_mse'],4),
                 'ΔAIC':round(r['delta_aic'],2),
                 'LRT_p':round(r['lrt_p'],4),
                 '†': '†' if r['weak_id'] else ''})

df_res = pd.DataFrame(rows)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
print(df_res.to_string(index=False))
df_res.to_csv('../data/insample_fit_results.csv', index=False)
print("\nSaved → data/insample_fit_results.csv")


## 2.3  Model selection summary

In [ ]:
n_pl  = sum(r['delta_aic'] < 0  for r in fits.values())
n_lrt = sum(r['lrt_p']    < 0.05 for r in fits.values())
print(f"Power law preferred (ΔAIC < 0) : {n_pl}/{len(fits)} funds")
print(f"LRT significant (p < 0.05)     : {n_lrt}/{len(fits)} funds")
print()
for f, r in sorted(fits.items(), key=lambda x: x[1]['delta_aic']):
    tag = 'PL preferred' if r['delta_aic'] < 0 else 'Inconclusive'
    wi  = ' †' if r['weak_id'] else ''
    print(f"  {f:<30} ΔAIC={r['delta_aic']:>7.2f}  p={r['lrt_p']:.4f}  {tag}{wi}")


## 2.4  Alpha bar chart with bootstrap CIs

In [ ]:
COLORS  = META['color'].to_dict()

sorted_fits = sorted(fits.items(), key=lambda x: x[1]['alpha'])
names  = [f for f,_ in sorted_fits]
alphas = np.array([r['alpha']   for _,r in sorted_fits])
ci_lo  = np.array([r['ci_lo']   for _,r in sorted_fits])
ci_hi  = np.array([r['ci_hi']   for _,r in sorted_fits])
cols   = [COLORS.get(r['strategy'],'gray') for _,r in sorted_fits]
weak   = [r['weak_id'] for _,r in sorted_fits]
y_pos  = np.arange(len(names))

err_lo = np.clip(alphas - ci_lo, 0, None)
err_hi = np.clip(ci_hi  - alphas, 0, None)

short = [n.replace(' Technologies','').replace(' Capital','')
          .replace(' Management','').replace(' Strategic','')
          .replace(' Investment','').replace(' Group','') for n in names]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(y_pos, alphas, color=cols, alpha=0.75, height=0.6)
ax.errorbar(alphas, y_pos, xerr=[err_lo, err_hi],
            fmt='none', color='k', capsize=4, lw=1.2)
for i, w in enumerate(weak):
    if w:
        ax.text(ci_hi[i]+0.05, y_pos[i], '†', va='center', fontsize=11, color='firebrick')

ax.set_yticks(y_pos); ax.set_yticklabels(short, fontsize=8)
ax.axvline(pool['alpha'], color='gray', ls=':', lw=1, label=f"Pooled α = {pool['alpha']:.2f}")
ax.axvline(1.0, color='k', ls=':', lw=1, label='α = 1  (linear)')
ax.set_xlabel('Scaling exponent  α̂  (± 95% bootstrap CI)')
ax.set_title('Fund-level Pareto exponents — in-sample fits')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/alpha_bootstrap_bars.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.5  Log-log scatter with per-fund fits

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for f, r in fits.items():
    strat = r['strategy']
    ax.scatter(r['aum'], r['hc'],
               color=COLORS.get(strat,'gray'), marker=MARKERS.get(strat,'o'),
               s=40, alpha=0.85, edgecolors='k', lw=0.3, label=f)
    aum_line = np.linspace(r['aum'].min(), r['aum'].max(), 80)
    ax.plot(aum_line, r['C']*aum_line**r['alpha'],
            color=COLORS.get(strat,'gray'), lw=0.8, ls='--', alpha=0.6)

# Pooled line
aum_all = np.concatenate([r['aum'] for r in fits.values()])
xl = np.linspace(aum_all.min(), aum_all.max(), 200)
ax.plot(xl, pool['C']*xl**pool['alpha'], 'k--', lw=1.8,
        label=f"Pooled  α={pool['alpha']:.2f}, R²={pool['R2']:.2f}")

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('AUM  (USD bn)'); ax.set_ylabel('Headcount')
ax.legend(fontsize=6, ncol=2)
ax.set_title('Log-log scatter — in-sample fits')
plt.tight_layout()
plt.savefig('../figures/loglog_insample.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.6  Residual diagnostics

In [ ]:
all_e   = np.concatenate([r['resid'] for r in fits.values()])
all_aum = np.concatenate([r['aum']   for r in fits.values()])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.scatter(all_aum, all_e, alpha=0.55, s=22, color='steelblue', edgecolors='none')
ax.axhline(0,    color='k',       lw=0.8)
ax.axhline( 0.3, color='#e6a817', ls='--', lw=1.1, label='±0.3 log-unit band')
ax.axhline(-0.3, color='#e6a817', ls='--', lw=1.1)
ax.set_xscale('log')
ax.set_xlabel('AUM  (USD bn)')
ax.set_ylabel('ε̂  =  ln Nₛᵒᵇˢ − ln Nₛᶠⁱᵗ')
ax.set_title('Residuals vs AUM')
ax.legend(fontsize=8)

ax = axes[1]
n  = len(all_e)
q  = np.sort(all_e)
th = stats.norm.ppf(np.linspace(1/(2*n), 1-1/(2*n), n))
ax.scatter(th, q, s=18, alpha=0.65, color='steelblue', edgecolors='none')
ax.plot([th.min(), th.max()],
        [th.min()*q.std()+q.mean(), th.max()*q.std()+q.mean()],
        'r--', lw=1.2)
rqq = np.corrcoef(th, q)[0,1]
ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Sample quantiles')
ax.set_title(f'Normal Q–Q  (r = {rqq:.3f})')

plt.tight_layout()
plt.savefig('../figures/residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
within = np.mean(np.abs(all_e) < 0.3)*100
print(f"{within:.1f}% of residuals within ±0.3 log-units")


## 2.7  Capital efficiency vs. alpha

In [ ]:
from scipy.stats import linregress

effs   = [r['efficiency'] for r in fits.values()]
alphas = [r['alpha']      for r in fits.values()]
cols_e = [COLORS.get(r['strategy'],'gray') for r in fits.values()]
fnames = list(fits.keys())

slope, intercept, rv, pv, _ = linregress(alphas, np.log(effs))
print(f"log(efficiency) ~ α :  r = {rv:.2f},  p = {pv:.4f}")

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(alphas, effs, c=cols_e, s=55, zorder=3, edgecolors='k', lw=0.4)
for a, e, nm in zip(alphas, effs, fnames):
    ax.annotate(nm.split()[0], (a, e), fontsize=7,
                xytext=(4,3), textcoords='offset points')
xl = np.linspace(min(alphas), max(alphas), 100)
ax.plot(xl, np.exp(intercept+slope*xl), 'k--', lw=1.2,
        label=f'OLS  r = {rv:.2f}')
ax.set_yscale('log')
ax.set_xlabel('Scaling exponent  α̂')
ax.set_ylabel('AUM per employee  (USD million)')
ax.set_title('Capital efficiency vs. scaling exponent')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../figures/efficiency_vs_alpha.png', dpi=150, bbox_inches='tight')
plt.show()
